# Comparación pareada de capacidad: RNN32 y RNN16 con F2

Este notebook vuelve a entrenar una referencia recurrente de **32 unidades** y después una candidata de **16 unidades** en la misma sesión. La candidata cambia solo `model.hidden_size`; embeddings de dimensión 4, rama estática de 16, cabeza de 16, W=8, datos, máscaras, semilla 42 y entrenamiento permanecen iguales.

Ambas corridas seleccionan checkpoint por **F2 positiva a umbral fijo 0,5**, con menor BCE de validación como desempate. Después seleccionan su umbral F2 con los pesos congelados. La pérdida diferenciable sigue siendo BCE sin ponderación. La corrida F1 anterior y la referencia F2 histórica se conservan.

El control principal es la referencia RNN32 reentrenada en esta misma pareja. `runs/rnn_v1_seed42`, asociada al código histórico `cfcfb95`, aporta contexto: si cambia el entorno, una diferencia frente a ella no puede atribuirse únicamente a capacidad.

Train sigue siendo 2022–2023 y validación 2024. **2025 permanece reservada.** El notebook se entrega sin ejecuciones ni resultados anticipados.

## 1. Preparar y cargar el paquete de la pareja

Desde la raíz del repositorio, con las dependencias de experimentos instaladas:

```bash
python scripts/empaquetar_experimento_rnn.py \
  --config configs/experimento_rnn16_v1.json \
  --output runs/rnn_colab_rnn16_v1.zip
```

La configuración candidata declara `companion_configs=['configs/experimento_rnn32_pareado_v1.json']`: el ZIP contiene ambas configuraciones, el código compartido y los artefactos congelados de train/validación. Guardar el SHA-256 mostrado por el empaquetador.

En Colab se recomienda **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**, si está disponible. CPU también permite una comparación válida si ambas corridas usan el mismo entorno y dispositivo. La pareja comprueba versiones, dispositivo, GPU/CUDA y opciones de determinismo antes, entre y después de los entrenamientos.

Subir el ZIP desde el panel **Archivos** a `/content/rnn_colab_rnn16_v1.zip`. La celda siguiente reutiliza ese archivo; solo llama a `files.upload()` si falta. Si tiene otro nombre, ajustar `PACKAGE_PATH`. Usar una sesión limpia: la extracción requiere que `/content/f1-rnn16` no exista. Drive es opcional.

In [ ]:
from pathlib import Path
import hashlib
import json
import stat
import subprocess
import sys
import zipfile

from google.colab import files

# Opcional: copiar aquí el SHA-256 calculado localmente.
EXPECTED_PACKAGE_SHA256 = ""
ROOT = Path("/content/f1-rnn16")
CONFIG_RELATIVE_PATH = Path("configs/experimento_rnn16_v1.json")
REFERENCE_CONFIG_RELATIVE_PATH = Path("configs/experimento_rnn32_pareado_v1.json")
# Si el ZIP ya está en el panel Archivos, se lee directamente sin volver a subirlo.
PACKAGE_PATH = Path("/content/rnn_colab_rnn16_v1.zip")

if ROOT.exists():
    raise FileExistsError("/content/f1-rnn16 ya existe. Usa una sesión limpia para esta corrida.")
if PACKAGE_PATH.is_file():
    print("Se reutiliza el ZIP ya subido:", PACKAGE_PATH)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube exactamente un archivo ZIP generado por el empaquetador RNN16.")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if Path(uploaded_name).suffix.lower() != ".zip":
        raise ValueError("El archivo debe ser el ZIP generado para el experimento pareado RNN16.")
    PACKAGE_PATH.write_bytes(uploaded_bytes)
    del uploaded_bytes, uploaded

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

package_sha256 = file_sha256(PACKAGE_PATH)
print("SHA-256 del paquete:", package_sha256)
if EXPECTED_PACKAGE_SHA256 and package_sha256 != EXPECTED_PACKAGE_SHA256.strip().lower():
    raise ValueError("El SHA-256 no coincide con el paquete local.")

# Validar todos los nombres antes de extraer: sin rutas que salgan del destino,
# enlaces simbólicos ni destinos repetidos.
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    destinations = set()
    for member in members:
        name = member.filename
        if "\\" in name:
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        candidate = Path(name)
        if candidate.is_absolute() or ".." in candidate.parts:
            raise ValueError(f"Ruta ZIP fuera del destino: {name!r}")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError(f"Enlace simbólico no admitido: {name!r}")
        destination = (ROOT / candidate).resolve()
        if not destination.is_relative_to(ROOT.resolve()) or destination == ROOT.resolve():
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        if destination in destinations:
            raise ValueError(f"Destino ZIP repetido: {name!r}")
        destinations.add(destination)
    ROOT.mkdir(parents=True)
    archive.extractall(ROOT)

# Comprobar el contrato y todos los hashes del paquete antes de instalar o
# importar su código. El manifiesto no se incluye en su propio mapa de hashes.
package_manifest_path = ROOT / "colab_package_manifest.json"
package_manifest = json.loads(package_manifest_path.read_text(encoding="utf-8"))
if package_manifest.get("artifact_version") != "rnn_colab_inputs_v1":
    raise ValueError("Versión de paquete Colab no admitida.")
if package_manifest.get("test_arrays_included") is not False:
    raise ValueError("El paquete debe declarar test_arrays_included=false.")
if package_manifest.get("experiment_config") != CONFIG_RELATIVE_PATH.as_posix():
    raise ValueError("El paquete no declara la configuración candidata RNN16 esperada.")
file_hashes = package_manifest.get("files_sha256")
if not isinstance(file_hashes, dict) or not file_hashes:
    raise ValueError("El manifiesto debe enumerar los archivos y sus SHA-256.")
reference_config_name = REFERENCE_CONFIG_RELATIVE_PATH.as_posix()
if package_manifest.get("companion_configs") != [reference_config_name]:
    raise ValueError("El paquete debe declarar la configuración RNN32 acompañante.")
if reference_config_name not in file_hashes:
    raise ValueError("Falta el hash de la configuración RNN32 acompañante.")
if package_manifest.get("companion_configs_sha256") != {reference_config_name: file_hashes[reference_config_name]}:
    raise ValueError("El hash de companion_configs no coincide con files_sha256.")
for name, expected_hash in file_hashes.items():
    if not isinstance(name, str) or "\\" in name:
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    relative = Path(name)
    resolved = (ROOT / relative).resolve()
    if relative.is_absolute() or ".." in relative.parts or not resolved.is_relative_to(ROOT.resolve()):
        raise ValueError(f"Ruta de manifiesto fuera del destino: {name!r}")
    if resolved == ROOT.resolve() or resolved == package_manifest_path.resolve():
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    if not isinstance(expected_hash, str) or len(expected_hash) != 64:
        raise ValueError(f"SHA-256 no válido para {name!r}")
    if any(character not in "0123456789abcdef" for character in expected_hash):
        raise ValueError(f"SHA-256 no válido para {name!r}")

expected_files = set(file_hashes) | {"colab_package_manifest.json"}
actual_files = {file.relative_to(ROOT).as_posix() for file in ROOT.rglob("*") if file.is_file()}
if actual_files != expected_files:
    raise ValueError(
        "Archivos distintos del manifiesto: "
        f"faltantes={sorted(expected_files - actual_files)}, "
        f"adicionales={sorted(actual_files - expected_files)}"
    )
for name, expected_hash in file_hashes.items():
    if file_sha256(ROOT / name) != expected_hash:
        raise ValueError(f"El SHA-256 no coincide para {name!r}")
print("Manifiesto y SHA-256 verificados:", len(file_hashes), "archivos; sin arrays de prueba.")

required = [
    "requirements-experiments.txt",
    CONFIG_RELATIVE_PATH.as_posix(),
    REFERENCE_CONFIG_RELATIVE_PATH.as_posix(),
    "scripts/entrenar_comparacion_capacidad.py",
    "scripts/entrenar_rnn.py",
    "scripts/evaluar_rnn.py",
    "src/f1_pitstop/rnn_model.py",
    "EDA/dataset/muestras_preprocesadas_v1_3/preprocessor.json",
    "EDA/dataset/muestras_preprocesadas_v1_3/manifest.json",
]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Paquete incompleto: {missing}")
print("Paquete extraído en", ROOT)


## 2. Dependencias, configuraciones y dispositivo

La instalación usa `requirements-experiments.txt` sin forzar wheels CPU ni fijar versiones históricas. El entorno histórico registró Python 3.13.16 y PyTorch 2.14.1+cu130; el entorno disponible en Colab puede ser diferente. El control de esta comparación es que **los dos entrenamientos actuales compartan el mismo entorno**, que queda registrado.

RNN16 significa 16 unidades en la celda recurrente. La rama estática sigue teniendo 16 unidades, la cabeza 16 y cada embedding dimensión 4. La representación mantiene 22 valores numéricos, 22 bits explícitos de disponibilidad y dos embeddings dinámicos: 52 entradas a la celda. No se incluyen conteos o fracciones de cobertura de QA.

Ambos modelos usan Adam con learning rate 0,001, BCE sin ponderación, batch 256, semilla 42, máximo 30 épocas, paciencia 6 y recorte de gradiente a 1. El máximo y la paciencia son comunes; la época seleccionada y la cantidad final de épocas pueden variar como resultado del experimento.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-experiments.txt")],
    check=True,
)


In [ ]:
from importlib import metadata

sys.path.insert(0, str(ROOT / "src"))
import torch
from f1_pitstop.rnn_model import SmallPitStopRNN

reference_config = json.loads((ROOT / REFERENCE_CONFIG_RELATIVE_PATH).read_text(encoding="utf-8"))
candidate_config = json.loads((ROOT / CONFIG_RELATIVE_PATH).read_text(encoding="utf-8"))
print("Referencia pareada:")
print(json.dumps(reference_config, ensure_ascii=False, indent=2))
print("Candidata:")
print(json.dumps(candidate_config, ensure_ascii=False, indent=2))
print("Código compartido:", SmallPitStopRNN.__module__ + "." + SmallPitStopRNN.__name__)
print("Python:", sys.version)
for dependency in ("torch", "numpy", "pandas"):
    print(dependency + ":", metadata.version(dependency))
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU actual:", torch.cuda.get_device_name(0))
else:
    print("Se entrenarán ambos modelos en CPU. La comparación pareada sigue permitida.")
print("El runner verificará el entorno antes, entre y después de los entrenamientos.")


## 3. Entrenar la referencia y la candidata con el comando común

Se ejecutan **dos entrenamientos nuevos**, en orden: `rnn32_reference` y `rnn16_candidate`. El runner verifica que los controles materiales coincidan y que la única diferencia de arquitectura sea hidden size 32 → 16. El preprocesador ya ajustado con 46.162 muestras elegibles de train se reutiliza; no se ejecuta `fit` ni se transforma prueba.

Cada checkpoint maximiza F2 positiva de validación a umbral **fijo 0,5**, con menor BCE como desempate. La parada temprana observa ese criterio con paciencia 6. Después, con los pesos elegidos congelados, se maximiza F2 sobre `0,05, 0,06, …, 0,95`; los empates se resuelven por mayor precision y luego mayor umbral. F2 es el criterio de selección; la pérdida sigue siendo BCE.

El runner compara el entorno antes, entre y después, y registra también el entorno de cada hijo. Si no se cumplen los controles de la pareja, la comparación no se autoriza. No cambiar el tipo de entorno ni instalar paquetes entre entrenamientos.

La carpeta de salida debe ser nueva. Se genera un ZIP padre con ambos modelos y los manifiestos de la pareja, sin ZIPs hijos anidados.

In [ ]:
RUN_DIR = ROOT / "runs/rnn16_vs_rnn32_f2_v1_seed42"
RUN_ZIP = RUN_DIR.with_suffix(".zip")
if RUN_DIR.exists() or RUN_ZIP.exists():
    raise FileExistsError("La salida de esta pareja ya existe. Conserva sus artefactos antes de repetir.")

subprocess.run(
    [
        sys.executable,
        "scripts/entrenar_comparacion_capacidad.py",
        "--reference-config", REFERENCE_CONFIG_RELATIVE_PATH.as_posix(),
        "--candidate-config", CONFIG_RELATIVE_PATH.as_posix(),
        "--output", str(RUN_DIR.relative_to(ROOT)),
    ],
    cwd=ROOT,
    check=True,
)


## 4. Revisar entorno, resultados y curvas de ambos modelos

Primero revisar `pair_environment.json`: deben existir snapshots antes, entre y después, y `comparison_authorized=true`. El resumen compara RNN32 y RNN16 entrenadas en esta sesión. No se presupone que RNN16 mejore F2 ni que menos capacidad elimine la brecha entre train y validación.

Revisar F2, F1, precision, recall, AP, BCE, época elegida y umbral; conservar métricas a 0,5 y al umbral elegido para ambos modelos. Los historiales usan el mismo protocolo. Con los vocabularios publicados, los tamaños esperados son 4.121 parámetros para RNN32 y 2.233 para RNN16; los conteos de los modelos instanciados quedan en el resumen.

Estas métricas se calculan por muestra piloto-vuelta. La cobertura por evento se calcula después, únicamente en el evaluador local, con eventos e instantes de decisión verificados por hash. Validación se usa para seleccionar pesos y umbrales; una semilla no estima variabilidad y estos resultados no sustituyen una evaluación final independiente.

In [ ]:
import pandas as pd
from IPython.display import display

pair_environment = json.loads((RUN_DIR / "pair_environment.json").read_text(encoding="utf-8"))
print("Entorno y controles de la pareja:")
print(json.dumps(pair_environment, ensure_ascii=False, indent=2))
if pair_environment.get("comparison_authorized") is not True:
    raise ValueError("Los controles de entorno no autorizan comparar esta pareja.")

pair_summary = json.loads((RUN_DIR / "pair_summary.json").read_text(encoding="utf-8"))
print("Resumen de comparación:")
print(json.dumps(pair_summary, ensure_ascii=False, indent=2))
for child in ("rnn32_reference", "rnn16_candidate"):
    print("Historial:", child)
    display(pd.read_csv(RUN_DIR / child / "history.csv"))
print("Artefactos de pareja:")
for artifact in sorted(RUN_DIR.iterdir()):
    print(" -", artifact.name)


## 5. Descargar la pareja completa

El ZIP padre contiene `rnn32_reference/`, `rnn16_candidate/`, `pair_summary.json`, `pair_environment.json`, `pair_manifest.json` y el estado de la ejecución. Cada hijo conserva pesos, arquitectura, configuración, preprocesador, umbral, historial, entorno, métricas y predicciones identificables por `sample_id`.

La descarga usa el ZIP producido por el runner; no reconstruye artefactos ni incorpora ZIPs hijos. Conservarlo antes de cerrar Colab.

In [ ]:
if not RUN_ZIP.is_file():
    raise FileNotFoundError(f"El entrenamiento no generó el ZIP esperado: {RUN_ZIP}")
print("Descarga:", RUN_ZIP)
files.download(str(RUN_ZIP))


### Copia opcional en Google Drive

La celda está desactivada por defecto. Para conservar una copia adicional del ZIP padre, cambiar `COPY_TO_DRIVE` a `True`; Colab solicitará acceso al montar Drive. No hace falta para entrenar, descargar ni evaluar localmente.

In [ ]:
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    drive_directory = Path("/content/drive/MyDrive/F1-PitStop-Predictor")
    drive_directory.mkdir(parents=True, exist_ok=True)
    drive_target = drive_directory / RUN_ZIP.name
    if drive_target.exists():
        raise FileExistsError(f"Ya existe una copia: {drive_target}")
    shutil.copy2(RUN_ZIP, drive_target)
    print("Copia conservada en", drive_target)


## 6. Evaluar ambos hijos localmente en CPU

Extraer el ZIP padre en una carpeta nueva del repositorio. Cada `--bundle` debe apuntar a la carpeta hija que contiene `weights.pt`. Desde la raíz del repositorio:

```bash
python scripts/evaluar_rnn.py \
  --bundle runs/rnn16_vs_rnn32_f2_v1_seed42/rnn32_reference \
  --data EDA/dataset/muestras_preprocesadas_v1_3 \
  --output eval/rnn16_vs_rnn32_f2_v1_seed42/rnn32_reference \
  --device cpu

python scripts/evaluar_rnn.py \
  --bundle runs/rnn16_vs_rnn32_f2_v1_seed42/rnn16_candidate \
  --data EDA/dataset/muestras_preprocesadas_v1_3 \
  --output eval/rnn16_vs_rnn32_f2_v1_seed42/rnn16_candidate \
  --device cpu
```

Ajustar las rutas `--bundle` si el ZIP se extrae en otra ubicación. Ambos evaluadores usan la misma validación 2024 y sus respectivos pesos y umbrales congelados. Contrastan probabilidades Colab/CPU por `sample_id` con `parity_atol=1e-5` y `parity_rtol=1e-4`. Las probabilidades próximas al umbral pueden cambiar de decisión dentro de la tolerancia; se registran las diferencias. Una discrepancia fuera de tolerancia debe investigarse antes de publicar métricas.

La evaluación local necesita además las fuentes ya existentes `events.csv` de `etiquetado_pit_entry_v3_2022_2025` y `sample_index.csv.gz` de `muestras_causales_v1_3`, en las rutas fijadas en configuración y verificadas por hash. No se incluyen en Colab. La cobertura por evento se calcula solo aquí.

Comparar primero la pareja actual, conservando el resultado negativo si lo hubiera. La referencia histórica aporta contexto y su entorno no se supone idéntico al actual. **2025 sigue reservada.**

Contrato: `docs/specs/13_experimento_capacidad_rnn16.md`.